# Warm start on the IDAES CSTR

Under closed-loop control the next problem is the last one moved one
step, so the last solution moved one step is nearly its answer.
`drto.warm_start_dynamic` shifts it. This notebook runs one loop
iteration on the pattern a loop actually uses: one persistent model,
built and scaled once, then solved, shifted, and solved again, the
second solve told only that the barrier may start small. The shift
carries values only; the solver rebuilds its own multipliers from a
good starting point in its first iterations, faster than any
multipliers we could hand it. The solver is pounce, which reads the
scaling suffix and the option through its standard interface.

## One model, built and scaled once

Declarations, the setpoint, the terminal segment, the cold start with
the flowsheet's magnitudes as its `scale` source, energy in joules near
`1e7` and duties in watts near `1e6`, then the assembly and
`drto.scale` writing the same magnitudes for the solves. The whole
loop keeps this one model, and every solve receives the factors under
`nlp_scaling_method=user-scaling`.

In [1]:
import contextlib, io, time

import pyomo.environ as pyo
from pyomo.contrib.solver.common.factory import SolverFactory

import drto
from models.idaes_cstr import DC_START, F_IN, VOLUME, build

m = build()

ss = pyo.TransformationFactory("drto.steady_state_simulation").create_using(
    m, controls={m.fs.cstr.control_volume.heat.name: 0.0,
                 m.fs.cstr.inlet.flow_vol.name: F_IN})
drto.initialize_steady_state(ss)
drto.scaled_solve(ss)
cvs = ss.fs.cstr.control_volume
for j, ssp in (("NaOH", m.ss_naoh), ("EthylAcetate", m.ss_ea),
               ("SodiumAcetate", m.ss_sa), ("Ethanol", m.ss_etoh)):
    ssp.set_value(pyo.value(cvs.material_holdup["Liq", j]))
for j, sgn in (("NaOH", 1), ("EthylAcetate", 1),
               ("SodiumAcetate", -1), ("Ethanol", -1)):
    m.mat0[j] = pyo.value(cvs.material_holdup["Liq", j]) + sgn * DC_START * VOLUME
for k in m.eng_ss:
    m.eng_ss[k] = pyo.value(cvs.energy_holdup[k])

pyo.TransformationFactory("drto.infinite_horizon").apply_to(m)
drto.cold_start_dynamic(m, profile="exponential", time_constant=3.0,
                        scale={"J": 1e7, "W": 1e6})
pyo.TransformationFactory("drto.dynamic_optimization").apply_to(m)

drto.scale(m, source={"J": 1e7, "W": 1e6})
res = SolverFactory("pounce").solve(
    m,
    solver_options={"nlp_scaling_method": "user-scaling", "mu_init": 1e-6},
    tee=True)
print(res.termination_condition.name)

component keys that are not exported as part of the NL file.  Skipping.


component keys that are not exported as part of the NL file.  Skipping.


component keys that are not exported as part of the NL file.  Skipping.


********************************************************************************

                    ####    ###   /   # /#   #/  ####  #####
                    #   #  #   # /#   #/ ##  /  #      #
                    ####   #   #/ #   /  # #/#  #      ####
                    #      #   /  #  /#  # /##  #      #
                    #       ##/    #/#   #/  #   ####  #####

********************************************************************************
This program contains POUNCE, a pure-Rust interior-point optimization solver
for nonlinear, conic, and global problems (its NLP core is ported from Ipopt).
Released under the Eclipse Public License (EPL) — drop-in compatible with Ipopt.
         For more information visit https://github.com/jkitchin/pounce
********************************************************************************

This is POUNCE version 0.10.0, running with linear solver FERAL.



Number of nonzeros in equality constraint Jacobian...:     4472
Number of nonzeros in inequality constraint Jacobian.:       20
Number of nonzeros in Lagrangian Hessian.............:      645

Total number of variables............................:     1414
                     variables with only lower bounds:      342
                variables with lower and upper bounds:       92
                     variables with only upper bounds:        0
Total number of equality constraints.................:     1343
Total number of inequality constraints...............:        4
        inequality constraints with only lower bounds:        4
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter      objective   inf_pr   inf_du lg(mu)    ||d|| lg(rg) alpha_du alpha_pr  ls
   0  6.7507105e+03 1.59e+07 1.02e+03   -6.0 0.00e+00      - 0.00e+00 0.00e+00   0
   1 -1.5250645e+03 5.57e+07 9.98e+02   -6.0 1.90e+06      - 1.5

   7  1.7506174e+03 2.30e+07 6.83e+02   -6.0 1.00e+05      - 5.75e-01 9.95e-01h  1
   8  3.6410885e+03 5.15e+06 1.26e+02   -6.0 3.85e+04      - 4.89e-01 1.00e+00H  1
   9  3.4541118e+03 1.49e+06 3.63e+01   -6.0 5.73e+03      - 7.98e-01 1.00e+00f  1
iter      objective   inf_pr   inf_du lg(mu)    ||d|| lg(rg) alpha_du alpha_pr  ls
  10  3.5360351e+03 3.25e+04 1.13e+00   -6.0 3.98e+02      - 9.83e-01 1.00e+00h  1
  11  3.5376311e+03 2.45e+00 1.98e-04   -6.0 1.31e+02      - 1.00e+00 1.00e+00h  1
  12  3.5376313e+03 2.52e-02 2.56e-10   -6.0 7.21e+05      - 1.00e+00 1.00e+00h  1
  13  3.5376313e+03 2.48e-05 3.07e-11   -9.0 5.54e+06      - 1.00e+00 1.00e+00h  1


Number of Iterations....: 13

                                   (scaled)                 (unscaled)
Objective...............:   3.5376313199799847e+03    3.5376313199799847e+03
Dual infeasibility......:   3.0659402850314868e-11    3.0659402850314868e-11
Constraint violation....:   1.0004441719502211e-11    2.4795532226562500e-05
Va

convergenceCriteriaSatisfied


## One step later: shift everything

The loop implements the first move and the state advances one sample;
the model's own solution at t = h stands in for the measurement, read
directly, since the model never leaves its own units. The shift moves
every variable one sampling time forward.

In [2]:
cv = m.fs.cstr.control_volume
h = 1.0
for j in ("NaOH", "EthylAcetate", "SodiumAcetate", "Ethanol"):
    m.mat0[j] = pyo.value(cv.material_holdup[h, "Liq", j])
m.eng0["Liq"] = pyo.value(cv.energy_holdup[h, "Liq"])

print(drto.warm_start_dynamic(m))

drto warm_start_dynamic (the previous solution, one step on)
  shift         : 1 time units
  copied        : 1009 values on aligned points
  interpolated  : 496 values between points
  filled        : 0 values past the end
  tail          : shifted through t = tN + atanh(tau)/gamma


## The second solve, warm

The same model again from the shifted values, with the same scaling
and `mu_init=1e-6`: the shifted point is nearly optimal, so the
barrier starts where it would otherwise have to work its way down to,
and the solver rebuilds its multipliers from the point and stops.

In [3]:
res = SolverFactory("pounce").solve(
    m,
    solver_options={"nlp_scaling_method": "user-scaling", "mu_init": 1e-6},
    tee=True)
print(res.termination_condition.name)

component keys that are not exported as part of the NL file.  Skipping.


********************************************************************************

                    ####    ###   /   # /#   #/  ####  #####
                    #   #  #   # /#   #/ ##  /  #      #
                    ####   #   #/ #   /  # #/#  #      ####
                    #      #   /  #  /#  # /##  #      #
                    #       ##/    #/#   #/  #   ####  #####

********************************************************************************
This program contains POUNCE, a pure-Rust interior-point optimization solver
for nonlinear, conic, and global problems (its NLP core is ported from Ipopt).
Released under the Eclipse Public License (EPL) — drop-in compatible with Ipopt.
         For more information visit https://github.com/jkitchin/pounce
********************************************************************************

This is POUNCE version 0.10.0, running with linear solver FERAL.



Number of nonzeros in equality constraint Jacobian...:     4472
Number of nonzeros in inequality constraint Jacobian.:       20
Number of nonzeros in Lagrangian Hessian.............:      645

Total number of variables............................:     1414
                     variables with only lower bounds:      342
                variables with lower and upper bounds:       92
                     variables with only upper bounds:        0
Total number of equality constraints.................:     1343
Total number of inequality constraints...............:        4
        inequality constraints with only lower bounds:        4
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter      objective   inf_pr   inf_du lg(mu)    ||d|| lg(rg) alpha_du alpha_pr  ls
   0  1.6659160e+02 1.71e+07 1.02e+03   -6.0 0.00e+00      - 0.00e+00 0.00e+00   0
   1  6.5530935e+01 1.68e+07 9.61e+02   -6.0 1.83e+04      - 8.0

   7  6.5591602e+01 1.43e-06 4.43e-12   -9.0 5.54e+06      - 1.00e+00 1.00e+00h  1


Number of Iterations....: 7

                                   (scaled)                 (unscaled)
Objective...............:   6.5591601659587170e+01    6.5591601659587170e+01
Dual infeasibility......:   4.4308803873868225e-12    4.4308803873868225e-12
Constraint violation....:   1.8189894035458565e-12    1.4305114746093752e-06
Variable bound violation:   0.0000000000000000e+00    0.0000000000000000e+00
Complementarity.........:   1.0014409265955031e-09    1.0014409265955031e-09
Overall NLP error.......:   1.0014409265955031e-09    1.4305114746093752e-06


Number of objective function evaluations             = 8
Number of objective gradient evaluations             = 8
Number of equality constraint evaluations            = 8
Number of inequality constraint evaluations          = 8
Number of equality constraint Jacobian evaluations   = 8
Number of inequality constraint Jacobian evaluations = 8
Number of

convergenceCriteriaSatisfied


The cold solve above took thirteen iterations and the warm-started
one takes seven: the shifted solution is nearly the answer, and the
solve rebuilds its multipliers from it and stops. That is warm
starting a receding horizon, whole.